In [51]:

import pandas as pd
import os

# Universal file loader function
def load_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == '.csv':
        df = pd.read_csv(file_path)
    elif ext in ['.xls', '.xlsx']:
        df = pd.read_excel(file_path, engine='openpyxl')
    elif ext == '.json':
        df = pd.read_json(file_path)
    elif ext == '.parquet':
        df = pd.read_parquet(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    print(f" Loaded file: {file_path}")
    print(f"Shape: {df.shape}")
    print("\nDynamic Attribute Summary:")
    for col in df.columns:
        dtype = df[col].dtype
        sample_values = df[col].dropna().head(3).tolist()
        print(f"Column: {col} | Type: {dtype} | Sample: {sample_values}")

    return df

# Example usage
file_path = r"synthetic_data_final.csv"  # Replace with your file path
df = load_file(file_path)  # df now holds your dataset


 Loaded file: synthetic_data_final.csv
Shape: (100000, 76)

Dynamic Attribute Summary:
Column: member_id | Type: object | Sample: ['MEM07757', 'MEM03032', 'MEM04897']
Column: patient_nbr | Type: object | Sample: ['PAT16943', 'PAT21600', 'PAT14776']
Column: age | Type: int64 | Sample: [48, 89, 35]
Column: sex | Type: object | Sample: ['Female', 'Female', 'Female']
Column: weight | Type: int64 | Sample: [185, 145, 299]
Column: bmi | Type: float64 | Sample: [38.4, 32.2, 23.2]
Column: smoker | Type: bool | Sample: [True, False, True]
Column: region | Type: object | Sample: ['Northeast', 'Southwest', 'Southwest']
Column: race | Type: object | Sample: ['Hispanic', 'Asian', 'AfricanAmerican']
Column: provider_id | Type: object | Sample: ['PROV8247', 'PROV4276', 'PROV3801']
Column: npi_number | Type: int64 | Sample: [7364524472, 6816967247, 1015759760]
Column: speciality | Type: object | Sample: ['General Practice', 'Internal Medicine', 'Internal Medicine']
Column: provider_organization_name |

In [52]:
df.isnull().sum()

member_id                  0
patient_nbr                0
age                        0
sex                        0
weight                     0
                          ..
metformin-rosiglitazone    0
metformin-pioglitazone     0
change                     0
diabetesMed                0
readmitted                 0
Length: 76, dtype: int64

In [53]:
import hashlib
# Helper function for hashing IDs
def hash_id(x):
    return hashlib.sha256(str(x).encode()).hexdigest()


In [54]:

# Create dim_patient from original dataset
dim_patient = df[['member_id', 'age', 'sex', 'region']]

# Add surrogate key
# dim_patient['member_sk'] = range(1, len(dim_patient) + 1)

# Hash patient_id for privacy
def hash_id(x):
    return hash(str(x))

dim_patient = df[df['age'] > 0].copy()
dim_patient['member_sk'] = range(1, len(dim_patient) + 1)
dim_patient['member_id_hashed'] = dim_patient['member_id'].apply(hash_id)


# dim_patient['patient_id_hashed'] = dim_patient['patient_id'].apply(hash_id)

# Select final columns
dim_patient = dim_patient[['member_sk', 'member_id_hashed', 'age', 'sex', 'region']]
dim_patient.to_csv('dim_patient.csv', index=False)
# Preview
print(dim_patient.head())
dim_patient.shape

   member_sk     member_id_hashed  age     sex     region
0          1   891942215582388005   48  Female  Northeast
1          2  7137451775312316309   89  Female  Southwest
2          3   815331141831290828   35  Female  Southwest
3          4  1807489755004826297   76  Female  Southeast
4          5  8989932288088514793   53  Female  Southeast


(100000, 5)

In [56]:

# Fix column names based on your dataset
dim_provider = df[['provider_id', 'speciality', 'npi_number']].drop_duplicates().reset_index(drop=True)

# Create surrogate key
dim_provider['provider_sk'] = range(1, len(dim_provider) + 1)

# Handle missing values
dim_provider['provider_id'] = dim_provider['provider_id'].fillna(-1)
dim_provider['npi_number'] = dim_provider['npi_number'].fillna(-1)
dim_provider['speciality'] = dim_provider['speciality'].fillna("Unknown")

# Reorder columns
dim_provider = dim_provider[['provider_sk', 'provider_id', 'speciality', 'npi_number']]

# Add default Unknown provider if needed
default_provider_sk = 0
if -1 in dim_provider['provider_id'].values:
    unknown_row = pd.DataFrame(
        [[default_provider_sk, -1, 'Unknown', -1]],
        columns=['provider_sk', 'provider_id', 'speciality', 'npi_number']
    )
    dim_provider = pd.concat([unknown_row, dim_provider], ignore_index=True)

# Save
dim_provider.to_csv('dim_provider.csv', index=False)

print(dim_provider.head())
dim_provider.shape



   provider_sk provider_id         speciality  npi_number
0            1    PROV8247   General Practice  7364524472
1            2    PROV4276  Internal Medicine  6816967247
2            3    PROV3801  Internal Medicine  1015759760
3            4    PROV2586        Orthopedics  9648584337
4            5    PROV6940   General Practice  6652633449


(99995, 4)

In [57]:

total_rows = len(dim_provider['provider_id'])
unique_rows = len(dim_provider['provider_id'].unique())

print(f"Total rows: {total_rows}")
print(f"Unique provider IDs: {unique_rows}")

if unique_rows < total_rows:
    print(" There are duplicates.")
else:
    print(" No duplicates.")


Total rows: 99995
Unique provider IDs: 10000
 There are duplicates.


In [59]:
import pandas as pd

# Load datasets
claims_df = pd.read_csv("synthetic_data_final.csv")
cpt_df = pd.read_csv("cpt4.csv")
icd_df = pd.read_csv("icd10_mapped_output.csv")

# --- STEP 0: Standardize procedure_code as STRING everywhere ---
claims_df['procedure_code'] = claims_df['procedure_code'].astype(str)
cpt_df['com.medigy.persist.reference.type.clincial.CPT.code'] = cpt_df['com.medigy.persist.reference.type.clincial.CPT.code'].astype(str)
icd_df['ICD-10 Code'] = icd_df['ICD-10 Code'].astype(str)

# --- Rename columns ---
cpt_df = cpt_df.rename(columns={
    'com.medigy.persist.reference.type.clincial.CPT.code': 'procedure_code',
    'label': 'cpt_description'
})
icd_df = icd_df.rename(columns={
    'ICD-10 Code': 'procedure_code',
    'ICD Description': 'icd_description'
})

# --- STEP 1: Extract procedure data ---
dim_procedure = claims_df[['procedure_code', 'procedure_description', 'category']].copy()

# --- STEP 2: Merge CPT descriptions ---
dim_procedure = dim_procedure.merge(
    cpt_df[['procedure_code', 'cpt_description']],
    on='procedure_code',
    how='left'
)

# --- STEP 3: Merge ICD descriptions ---
dim_procedure = dim_procedure.merge(
    icd_df[['procedure_code', 'icd_description']],
    on='procedure_code',
    how='left'
)

# --- STEP 4: Choose best description ---
dim_procedure['final_description'] = (
    dim_procedure['procedure_description']
        .fillna(dim_procedure['cpt_description'])
        .fillna(dim_procedure['icd_description'])
        .fillna("Unknown Procedure")
)

# --- STEP 5: Fill missing category ---
dim_procedure['category'] = dim_procedure['category'].fillna("Uncategorized")

# --- STEP 6: Remove duplicates ---
dim_procedure = dim_procedure[['procedure_code', 'final_description', 'category']].drop_duplicates()

# --- STEP 7: Create surrogate key ---
dim_procedure['proc_sk'] = range(1, len(dim_procedure) + 1)

# --- Final order ---
dim_procedure = dim_procedure[['proc_sk', 'procedure_code', 'final_description', 'category']]

# Save output
dim_procedure.to_csv("dim_procedure.csv", index=False)

dim_procedure.head()


,proc_sk,procedure_code,final_description,category
0,1,P0287,X-Ray,Laboratory
1,2,P0180,Office Visit,Primary Care
2,3,P0299,Surgery,Specialty Care
3,4,P0112,Consultation,Surgery
4,5,P0318,Physical Exam,Specialty Care


In [60]:
print(cpt_df.columns)
print(icd_df.columns)


Index(['procedure_code', 'cpt_description'], dtype='object')
Index(['Diagnosis', 'procedure_code', 'icd_description', 'Similarity Score',
       'Justification', 'Alternative Suggestions', 'Needs Review'],
      dtype='object')


In [61]:

df.head()

,member_id,patient_nbr,age,sex,weight,bmi,smoker,region,race,provider_id,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,MEM07757,PAT16943,48,Female,185,38.4,True,Northeast,Hispanic,PROV8247,...,Steady,Up,No,No,Down,Steady,Down,Yes,Yes,NO
1,MEM03032,PAT21600,89,Female,145,32.2,False,Southwest,Asian,PROV4276,...,Up,No,Steady,Steady,Up,Up,Steady,No,No,NO
2,MEM04897,PAT14776,35,Female,299,23.2,True,Southwest,AfricanAmerican,PROV3801,...,Down,Up,Down,Down,No,Down,Down,No,No,>30
3,MEM25242,PAT36900,76,Female,259,30.7,True,Southeast,Other,PROV2586,...,Down,Down,No,Up,Up,No,Steady,No,Yes,>30
4,MEM20795,PAT51367,53,Female,165,22.5,False,Southeast,Other,PROV6940,...,Down,Down,Down,No,Down,Up,Up,No,No,NO


In [63]:

# import pandas as pd

# # Assuming df is your main claims dataset
# # Load updated dataset with age categories
# df = pd.read_csv("synthetic_data_2025-12-05.csv")

# --- Create fact_claim ---
fact_claim = df[['member_id', 'provider_id', 'procedure_code', 'date_of_service',
                 'billed_amount', 'paid_amount', 'adjudication_status']].copy()

fact_claim['procedure_code'] = fact_claim['procedure_code'].astype(str)
dim_procedure['procedure_code'] = dim_procedure['procedure_code'].astype(str)


# Add surrogate key for fact table
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

fact_claim['member_id_hashed'] = fact_claim['member_id'].apply(hash_id)
patient_map = dict(zip(dim_patient['member_id_hashed'], dim_patient['member_sk']))
fact_claim['member_sk'] = fact_claim['member_id_hashed'].map(patient_map)


# Map provider_sk from dim_provider
provider_map = dict(zip(dim_provider['provider_id'], dim_provider['provider_sk']))
fact_claim['provider_sk'] = fact_claim['provider_id'].map(provider_map)

# Map proc_sk from dim_procedure
proc_map = dict(zip(dim_procedure['procedure_code'], dim_procedure['proc_sk']))
fact_claim['proc_sk'] = fact_claim['procedure_code'].map(proc_map)


# Drop original IDs for privacy concerns 
fact_claim = fact_claim[['claim_sk', 'member_sk', 'provider_sk', 'proc_sk',
                          'date_of_service', 'billed_amount', 'paid_amount', 'adjudication_status']]

# Validate
print(" fact_claim table created successfully!")
print(fact_claim.head())

# Save to CSV
fact_claim.to_csv('fact_claim.csv', index=False)


 fact_claim table created successfully!
   claim_sk  member_sk  provider_sk  proc_sk      date_of_service  \
0         1          1        98000    17055  2023-04-14T00:00:00   
1         2      42250        91065    17431  2022-03-21T00:00:00   
2         3          3        86384    17269  2022-05-14T00:00:00   
3         4      77985        90362    16812  2022-02-03T00:00:00   
4         5          5        74625    17036  2022-12-08T00:00:00   

   billed_amount  paid_amount adjudication_status  
0        2404.50      2111.13            Approved  
1        1568.54      1561.63            Approved  
2        1935.91      1479.05            Approved  
3        1078.94       780.64            Approved  
4        2071.38      1685.37            Approved  


In [64]:

# Add surrogate key first
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

# Now run validation
validation_results = []

# 1. Uniqueness Checks
checks = {
    'dim_patient.member_sk': dim_patient['member_sk'].is_unique,
    'dim_provider.provider_sk': dim_provider['provider_sk'].is_unique,
    'dim_procedure.proc_sk': dim_procedure['proc_sk'].is_unique,
    'fact_claim.claim_sk': fact_claim['claim_sk'].is_unique
}
for name, result in checks.items():
    validation_results.append({'Check': f'Unique {name}', 'Status': 'PASS' if result else 'FAIL'})

# 2. Null Checks in Critical Columns
critical_fact_cols = ['claim_sk', 'member_sk', 'provider_sk', 'proc_sk', 'date_of_service']
for col in critical_fact_cols:
    null_count = fact_claim[col].isnull().sum()
    validation_results.append({'Check': f'Nulls in fact_claim.{col}', 'Status': 'PASS' if null_count == 0 else f'FAIL ({null_count} nulls)'})

# 3. Referential Integrity Checks
invalid_patient_refs = fact_claim[~fact_claim['member_sk'].isin(dim_patient['member_sk'])]
validation_results.append({'Check': 'Referential integrity patient_sk', 'Status': 'PASS' if invalid_patient_refs.empty else f'FAIL ({len(invalid_patient_refs)} invalid)'})

invalid_provider_refs = fact_claim[~fact_claim['provider_sk'].isin(dim_provider['provider_sk'])]
validation_results.append({'Check': 'Referential integrity provider_sk', 'Status': 'PASS' if invalid_provider_refs.empty else f'FAIL ({len(invalid_provider_refs)} invalid)'})

invalid_proc_refs = fact_claim[~fact_claim['proc_sk'].isin(dim_procedure['proc_sk'])]
validation_results.append({'Check': 'Referential integrity proc_sk', 'Status': 'PASS' if invalid_proc_refs.empty else f'FAIL ({len(invalid_proc_refs)} invalid)'})

# Export validation report
report_df = pd.DataFrame(validation_results)
report_df.to_csv('validation_report.csv', index=False)

print(" Validation completed. Report saved as validation_report.csv")
print(report_df)


 Validation completed. Report saved as validation_report.csv
                                  Check Status
0          Unique dim_patient.member_sk   PASS
1       Unique dim_provider.provider_sk   PASS
2          Unique dim_procedure.proc_sk   PASS
3            Unique fact_claim.claim_sk   PASS
4          Nulls in fact_claim.claim_sk   PASS
5         Nulls in fact_claim.member_sk   PASS
6       Nulls in fact_claim.provider_sk   PASS
7           Nulls in fact_claim.proc_sk   PASS
8   Nulls in fact_claim.date_of_service   PASS
9      Referential integrity patient_sk   PASS
10    Referential integrity provider_sk   PASS
11        Referential integrity proc_sk   PASS


In [65]:

import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load your DataFrames into SQLite
dim_patient.to_sql('dim_patient', conn, index=False)
dim_provider.to_sql('dim_provider', conn, index=False)
dim_procedure.to_sql('dim_procedure', conn, index=False)
fact_claim.to_sql('fact_claim', conn, index=False)

# 1. Top Providers by Paid Amount

query1 = """
SELECT p.provider_id, p.speciality, SUM(f.paid_amount) AS total_paid
FROM fact_claim f
JOIN dim_provider p ON f.provider_sk = p.provider_sk
GROUP BY p.provider_id, p.speciality
ORDER BY total_paid DESC
LIMIT 10;
"""
result1 = pd.read_sql_query(query1, conn)
print("\nTop Providers by Paid Amount:")
print(result1)






Top Providers by Paid Amount:
  provider_id         speciality  total_paid
0    PROV7757         Pediatrics    31778.97
1    PROV5311        Orthopedics    30609.83
2    PROV5273          Neurology    29710.98
3    PROV9368        Orthopedics    29267.30
4    PROV4692  Internal Medicine    28428.96
5    PROV3714          Neurology    28368.36
6    PROV1619        Orthopedics    28174.92
7    PROV3840         Cardiology    28039.50
8    PROV1744   General Practice    27790.41
9    PROV3811          Neurology    27569.40


In [66]:

import os
import shutil
from datetime import datetime

source_file = 'fact_claim.csv'
landing_zone = 'landing/fact_claim/'
staging_zone = 'staging/fact_claim/'

# Create directories if they don't exist
os.makedirs(landing_zone, exist_ok=True)
os.makedirs(staging_zone, exist_ok=True)

# Step 1: Move file to landing zone
shutil.copy(source_file, landing_zone)

# Step 2: Log metadata
log_file = 'ingestion_log.txt'
with open(log_file, 'a') as log:
    log.write(f"{source_file}, {datetime.now()}, {os.path.getsize(source_file)} bytes\n")

# Step 3: Validate and move to staging
if os.path.exists(os.path.join(landing_zone, source_file)):
    shutil.move(os.path.join(landing_zone, source_file), staging_zone)
    print("File moved to staging successfully!")


Error: Destination path 'staging/fact_claim/fact_claim.csv' already exists

In [67]:
print(os.listdir('staging/fact_claim'))

['fact_claim.csv']


In [69]:

import pandas as pd

# Load the unified dataset
unified_df = pd.read_csv('synthetic_data_final.csv')

# Select relevant columns for ingestion layer fact_claim
fact_claim_ingestion = unified_df[[
    'member_id', 'provider_id', 'region', 'procedure_code', 'procedure_description',
    'date_of_service', 'pre_adjudication_date', 'adjudication_date', 'payment_date',
    'adjudication_status', 'billed_amount', 'paid_amount','sex','age','region','weight',
]].copy()

# Rename columns to match fact_claim structure
fact_claim_ingestion.rename(columns={
    'member_id': 'member_sk',
    'provider_id': 'provider_sk'
}, inplace=True)

# Add claim_sk as a sequential ID
fact_claim_ingestion.insert(0, 'claim_sk', range(1, len(fact_claim_ingestion) + 1))

# Save the enriched ingestion layer table
fact_claim_ingestion.to_csv('fact_claim_ingestion.csv', index=False)

print("Ingestion layer fact_claim table created successfully with new columns:")
print(fact_claim_ingestion.head())
print(f"Total rows: {len(fact_claim_ingestion)}")


Ingestion layer fact_claim table created successfully with new columns:
   claim_sk member_sk provider_sk     region procedure_code  \
0         1  MEM07757    PROV8247  Northeast          P0287   
1         2  MEM03032    PROV4276  Southwest          P0180   
2         3  MEM04897    PROV3801  Southwest          P0299   
3         4  MEM25242    PROV2586  Southeast          P0112   
4         5  MEM20795    PROV6940  Southeast          P0318   

  procedure_description      date_of_service pre_adjudication_date  \
0                 X-Ray  2023-04-14T00:00:00   2023-04-17T00:00:00   
1          Office Visit  2022-03-21T00:00:00   2022-03-24T00:00:00   
2               Surgery  2022-05-14T00:00:00   2022-05-15T00:00:00   
3          Consultation  2022-02-03T00:00:00   2022-02-08T00:00:00   
4         Physical Exam  2022-12-08T00:00:00   2022-12-12T00:00:00   

     adjudication_date         payment_date adjudication_status  \
0  2023-04-29T00:00:00  2023-05-10T00:00:00            Approv